# (gamma, T) Sweep — O-step Langevin Fock-PARFLM (OpenWebText)

**Purpose.** Phase-A *explore* run that scans the two — and only two — new
degrees of freedom introduced by the O-step Langevin retrofit:

- **`gamma`** — the friction of the exact Ornstein-Uhlenbeck (O) substep,
  which sets `c1 = exp(-gamma * dt)` (velocity retention per Verlet layer);
- **`T`** — the thermostat temperature, whose FDT-locked amplitude is
  `std = sqrt((T / m) * (1 - c1^2))`.

Everything else (architecture, `V_theta` / `V_phi` structure, xi channels,
read-out, register repulsion, WSD schedule) is held **identical to the
production notebook** `colab_fock_ostep_langevin_openwebtext.ipynb` via the
shared module `fock_ostep_setup.py`. That guarantees the `(gamma*, T*)` this
sweep selects transfers directly: you paste them into the production notebook's
config cell (`LANGEVIN_GAMMA`, `LANGEVIN_T`) and launch the long run.

**Workflow.**

```
this sweep notebook  ->  (gamma*, T*)  ->  paste into main notebook  ->  full O-step Langevin run
```

**Method.** For each grid point we train a *fresh* model for a short proxy
budget (`PROXY_STEPS`, default 8000) with the thermostat noise **on during
training** and **off at eval** (val PPL stays on the deterministic drift, the
same convention as the production run). We rank points by the val-PPL band
(median of the last few evals) among gradient-stable runs, tie-breaking toward
higher predictive entropy (the O-step is meant to *decompress* the over-committed
Verlet basins — see `Lessons_from_AlphaFold.md`) and then toward lower `gamma`.

**Cost note.** This is a retrain-per-point scan (the honest Phase-A: the noise
shapes the learned `V_theta`/`V_phi`, so a post-hoc eval cannot rank `T`).
Total ~ `len(GAMMA_GRID) * len(T_GRID) * PROXY_STEPS` optimiser steps. Start
with a small grid; widen only around the winner.

**Theory / provenance.**
- `companion_notes/Langevin_dynamics_reformulation_of_classical_damped_Lagrangian_flow.md`
  — §7 (O-step atom), §8 (retrofit assumptions/limits), §9 (accuracy curriculum
  with the exact (gamma, T) explore -> harvest -> exploit plan this notebook
  implements Phase A of).
- Paper section `sec:langevin-completion` (Thermal Langevin Completion).
- `companion_notes/Lessons_from_AlphaFold.md` — entropy / basin-coarseness lens.


In [ ]:
# ── Cell 1: Environment (clone repo, sys.path, drive, import module) ──
import os, sys, gc, shutil, subprocess, json, time, math
from pathlib import Path
import numpy as np

os.environ.setdefault('PYTORCH_ALLOC_CONF', 'expandable_segments:True')

REPO_URL    = 'https://github.com/dimitarpg13/semsimula-paper.git'
REPO_BRANCH = 'main'

IN_COLAB = 'google.colab' in sys.modules
print(f'IN_COLAB = {IN_COLAB}')


def _sh(cmd):
    print(f'$ {cmd}')
    r = subprocess.run(cmd, shell=True)
    if r.returncode != 0:
        raise RuntimeError(f'exit {r.returncode}: {cmd}')


# Sweep gets its OWN Drive dir so it never touches the production run's
# checkpoints; it only *reuses* the tokenised OpenWebText cache.
_GDRIVE_NAME = 'semsimula_fock_ostep_gammaT_sweep_owt'

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

    REPO_ROOT = Path('/content/semsimula-paper')
    if not (REPO_ROOT / '.git').exists():
        if REPO_ROOT.exists():
            shutil.rmtree(REPO_ROOT)
        _sh(f'git clone --depth 1 --branch {REPO_BRANCH} {REPO_URL} {REPO_ROOT}')
    else:
        try:
            _sh(f'git -C {REPO_ROOT} fetch --depth 1 origin {REPO_BRANCH}')
            _sh(f'git -C {REPO_ROOT} reset --hard origin/{REPO_BRANCH}')
        except RuntimeError as e:
            print(f'WARNING: repo refresh failed ({e}); using existing checkout.')

    GDRIVE_ROOT = Path(f'/content/drive/MyDrive/{_GDRIVE_NAME}')
    GDRIVE_ROOT.mkdir(parents=True, exist_ok=True)
    DATA_DIR = GDRIVE_ROOT / 'data'
    DATA_DIR.mkdir(exist_ok=True)
    repo_data = REPO_ROOT / 'notebooks' / 'conservative_arch' / 'data'
    if repo_data.is_symlink():
        repo_data.unlink()
    elif repo_data.is_dir():
        shutil.rmtree(repo_data)
    repo_data.symlink_to(DATA_DIR)
    RESULTS_DIR = GDRIVE_ROOT / 'results'
    RESULTS_DIR.mkdir(exist_ok=True)
    _sh('pip install -q transformers huggingface_hub pyarrow')
else:
    REPO_ROOT = Path('.').resolve()
    while not (REPO_ROOT / '.git').exists() and REPO_ROOT != REPO_ROOT.parent:
        REPO_ROOT = REPO_ROOT.parent
    DATA_DIR = REPO_ROOT / 'notebooks' / 'conservative_arch' / 'data'
    RESULTS_DIR = (REPO_ROOT / 'notebooks' / 'conservative_arch' / 'scaleup'
                   / 'results' / 'ostep_gammaT_sweep')
    for d in [DATA_DIR, RESULTS_DIR]:
        d.mkdir(parents=True, exist_ok=True)

CA_DIR = REPO_ROOT / 'notebooks' / 'conservative_arch'
for sub in ['', 'parf', 'multixi', 'scaleup', 'sarf_mass_variant', 'energetic_minima']:
    d = str(CA_DIR / sub) if sub else str(CA_DIR)
    if d not in sys.path:
        sys.path.insert(0, d)

import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# The shared setup module (lives in scaleup/, on sys.path). It mirrors the
# production notebook's config so the swept (gamma, T) transfers directly.
import fock_ostep_setup as fos
fos._self_test()

print(f'DEVICE      = {DEVICE}')
print(f'RESULTS_DIR = {RESULTS_DIR}')


In [ ]:
# ── Cell 2: Sweep configuration ──────────────────────────────────
# The two axes we scan. Keep the grid small first; widen around the winner.
GAMMA_GRID = [0.1, 0.3, 0.7]     # O-step friction (c1 = exp(-gamma*dt))
T_GRID     = [0.5, 1.0, 2.0]     # thermostat temperature (FDT amplitude)

# Proxy budget per grid point (the knob you asked to expose). 6-8k is enough
# to separate (gamma, T) by the val-PPL band while staying affordable.
PROXY_STEPS   = 8000

# Eval cadence for the proxy runs (cheaper than the production 500/40).
EVAL_INTERVAL = 1000
EVAL_ITERS    = 20
BAND_LAST_K   = 3                # summary band = median of last K evals

# Optimisation (mirrors the production run).
PEAK_LR       = 3e-4
WEIGHT_DECAY  = 0.01
GRAD_ACCUM    = 2
GRAD_CLIP     = 1.0
GRAD_CLIP_VPHI = 0.3
BATCH_SIZE    = None             # None -> auto-probe once, then reuse for all
SEED          = 0

# Base model config: IDENTICAL to colab_fock_ostep_langevin_openwebtext.ipynb.
# tie_T_to_beta MUST be False here so T actually varies across the grid.
BASE_CFG = fos.FockSetupConfig(
    langevin_ostep=True,
    langevin_tie_T_to_beta=False,   # let the sweep drive T
    langevin_noise_train=True,
    langevin_noise_eval=False,
)

print('Grid:', len(GAMMA_GRID), 'gammas x', len(T_GRID), 'T =',
      len(GAMMA_GRID) * len(T_GRID), 'points x', PROXY_STEPS, 'steps =',
      f'{len(GAMMA_GRID) * len(T_GRID) * PROXY_STEPS:,} optimiser steps total')
print('GAMMA_GRID =', GAMMA_GRID)
print('T_GRID     =', T_GRID)
print('Base variant tag (arch identity):', BASE_CFG.variant_tag())
print('  -> matches production notebook arch:',
      'xi5long_topk16_dt32da16_mh4_dcvt5x8_ob_untied_wsd_e5a_rep0.05'
      ' in tag ->',
      'xi5long_topk16_dt32da16_mh4_dcvt5x8_ob_untied_wsd_e5a_rep0.05'
      in BASE_CFG.variant_tag())


In [ ]:
# ── Cell 3: Data (reuse cached OpenWebText) + logfreq ────────────
from data_module import get_batch

MAX_TRAIN_TOKENS = 1_000_000_000
VAL_TOKENS       = 2_000_000
CHUNK_SIZE       = 50_000
VOCAB_SIZE       = 50257

train_cache = DATA_DIR / f'openwebtext_train_{MAX_TRAIN_TOKENS // 1_000_000}M.npy'
val_cache   = DATA_DIR / f'openwebtext_val_{VAL_TOKENS // 1_000_000}M.npy'

# Reuse the tokenised OWT cache from any prior run dir (esp. the production
# O-step run) instead of re-streaming ~1B tokens.
for alt_name in [
    'semsimula_fock_depthcond_vtheta_owt_xi5long_topk16_dt32da16_mh4_dcvt5x8_ob_untied_wsd',
    'semsimula_fock_structured_vtheta_owt_phase4',
    'semsimula_fock_gaussian_sarf_openwebtext_phase5',
    'semsimula_splm_openwebtext_phase4',
    'semsimula_splm_openwebtext_scaleup',
    'semsimula_parf_multixi_openwebtext_scaleup',
    'semsimula_fock_multixi_openwebtext_scaleup',
    'semsimula_splm_openwebtext',
    'semsimula_fock_multihead_openwebtext',
    'semsimula_fock_multicontext_vtheta_owt',
]:
    if train_cache.exists():
        break
    alt = (Path(f'/content/drive/MyDrive/{alt_name}/data') if IN_COLAB
           else Path.home() / alt_name / 'data')
    alt_train = alt / f'openwebtext_train_{MAX_TRAIN_TOKENS // 1_000_000}M.npy'
    alt_val   = alt / f'openwebtext_val_{VAL_TOKENS // 1_000_000}M.npy'
    if alt_train.exists():
        print(f'Reusing data cache from {alt}')
        shutil.copy2(str(alt_train), str(train_cache))
        shutil.copy2(str(alt_val), str(val_cache))
        break

if train_cache.exists() and val_cache.exists():
    print('Loading cached OpenWebText tokens ...')
    train_ids = np.load(str(train_cache))
    val_ids   = np.load(str(val_cache))
else:
    from transformers import AutoTokenizer
    from datasets import load_dataset
    tok = AutoTokenizer.from_pretrained('gpt2')
    print(f'Streaming OpenWebText ({MAX_TRAIN_TOKENS:,} train + {VAL_TOKENS:,} val) ...')
    ds = load_dataset('Skylion007/openwebtext', split='train', streaming=True,
                      trust_remote_code=True)
    all_ids, total, chunk_texts, n_docs, t0 = [], 0, [], 0, time.time()
    target = MAX_TRAIN_TOKENS + VAL_TOKENS
    for example in ds:
        chunk_texts.append(example['text']); n_docs += 1
        if len(chunk_texts) >= CHUNK_SIZE:
            all_ids.extend(tok.encode('\n\n'.join(chunk_texts)))
            total = len(all_ids); chunk_texts = []
            print(f'  {n_docs:,} docs  {total:,} tokens  ({time.time()-t0:.0f}s)', flush=True)
            if total >= target:
                break
    if chunk_texts:
        all_ids.extend(tok.encode('\n\n'.join(chunk_texts)))
    all_ids = np.array(all_ids, dtype=np.uint16)
    val_ids = all_ids[-VAL_TOKENS:]
    train_ids = all_ids[:-VAL_TOKENS][:MAX_TRAIN_TOKENS]
    del all_ids
    np.save(str(train_cache), train_ids)
    np.save(str(val_cache), val_ids)

print(f'train: {len(train_ids):,}   val: {len(val_ids):,}')

# logfreq surprisal (mass_mode='logfreq'); reuse repo/Drive copy or derive.
LOGFREQ_PATH  = CA_DIR / 'scaleup' / 'results' / 'logfreq_surprisal_openwebtext.npy'
DRIVE_LOGFREQ = RESULTS_DIR / 'logfreq_surprisal_openwebtext.npy'
if LOGFREQ_PATH.exists():
    LOGFREQ_FILE = LOGFREQ_PATH
elif DRIVE_LOGFREQ.exists():
    LOGFREQ_FILE = DRIVE_LOGFREQ
else:
    counts = np.bincount(train_ids.astype(np.int64), minlength=VOCAB_SIZE).astype(np.float64)
    p = (counts + 1.0) / (counts.sum() + VOCAB_SIZE)
    LOGFREQ_FILE = DRIVE_LOGFREQ
    LOGFREQ_FILE.parent.mkdir(parents=True, exist_ok=True)
    np.save(LOGFREQ_FILE, (-np.log(p)).astype(np.float32))
print(f'Logfreq: {LOGFREQ_FILE}')


In [ ]:
# ── Cell 4: Sweep runner (retrain a fresh model per (gamma, T)) ──
import itertools
import pandas as pd

RESULTS_CSV = RESULTS_DIR / 'gammaT_sweep_results.csv'
records = []

# Resume-safe: skip (gamma, T) points already in the CSV.
done = set()
if RESULTS_CSV.exists():
    _prev = pd.read_csv(RESULTS_CSV)
    records = _prev.to_dict('records')
    done = {(round(r['gamma'], 6), round(r['T'], 6)) for r in records}
    print(f'Resuming: {len(done)} points already done -> {sorted(done)}')

_batch = BATCH_SIZE
grid = list(itertools.product(GAMMA_GRID, T_GRID))
for gi, (gamma, T) in enumerate(grid):
    key = (round(gamma, 6), round(T, 6))
    if key in done:
        print(f'[{gi+1}/{len(grid)}] skip gamma={gamma} T={T} (done)')
        continue
    print(f'\n===== [{gi+1}/{len(grid)}] gamma={gamma}  T={T} '
          f'({PROXY_STEPS} steps) =====')
    t_pt = time.time()

    model, model_cfg = fos.build_fock_model(
        BASE_CFG, DEVICE, get_batch, train_ids, LOGFREQ_FILE,
        oom_probe=True, verbose=(gi == 0),
    )
    fos.init_output_bias(model, train_ids, VOCAB_SIZE, verbose=(gi == 0))
    fos.install_ostep(model, gamma=gamma, T=T,
                      noise_train=BASE_CFG.langevin_noise_train,
                      noise_eval=BASE_CFG.langevin_noise_eval,
                      verbose=True)

    if _batch is None:
        _batch = fos.auto_batch_size(model, get_batch, train_ids, DEVICE,
                                     BASE_CFG.block_size, GRAD_ACCUM)
    print(f'  batch={_batch} x accum={GRAD_ACCUM} (eff={_batch*GRAD_ACCUM})')

    optim = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=PEAK_LR, weight_decay=WEIGHT_DECAY, betas=(0.9, 0.95),
    )
    out = fos.train_proxy(
        model, BASE_CFG, model_cfg, optim, get_batch, train_ids, val_ids, DEVICE,
        steps=PROXY_STEPS, batch_size=_batch, grad_accum=GRAD_ACCUM,
        peak_lr=PEAK_LR, weight_decay=WEIGHT_DECAY,
        eval_interval=EVAL_INTERVAL, eval_iters=EVAL_ITERS, seed=SEED,
        lr_schedule='wsd', warmup_frac=0.05, stable_frac=0.60,
        grad_clip=GRAD_CLIP, grad_clip_vphi=GRAD_CLIP_VPHI,
        band_last_k=BAND_LAST_K, log_interval=500, verbose=True,
    )
    s = out['summary']
    rec = {'gamma': gamma, 'T': T, 'c1': math.exp(-gamma * BASE_CFG.dt),
           'best_ppl': s['best_ppl'], 'band_ppl': s['band_ppl'],
           'final_ppl': s['final_ppl'], 'final_entropy': s['final_entropy'],
           'grad_finite_frac': s['grad_finite_frac'], 'steps': PROXY_STEPS,
           'batch': _batch, 'minutes': (time.time() - t_pt) / 60.0}
    records.append(rec)
    pd.DataFrame(records).to_csv(RESULTS_CSV, index=False)
    print(f'  -> band_ppl={rec["band_ppl"]:.2f}  best={rec["best_ppl"]:.2f}  '
          f'ent={rec["final_entropy"]:.3f}  stable={rec["grad_finite_frac"]:.2f}  '
          f'({rec["minutes"]:.1f} min)')

    del model, optim
    gc.collect()
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()

print(f'\nSweep complete. Results -> {RESULTS_CSV}')


In [ ]:
# ── Cell 5: Results, selection, and the values to paste into main ──
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv(RESULTS_CSV)

# Selection: among gradient-stable points (finite grads all the way),
# lowest val-PPL band; tie-break toward higher entropy (decompressed basins,
# per Lessons_from_AlphaFold.md), then toward lower gamma (cheaper friction).
STABLE_MIN = 0.98        # require >=98% finite-grad steps
band_min = df['band_ppl'].min()
TIE_BAND = 0.01 * band_min   # points within 1% of the best band are "tied"

stable = df[df['grad_finite_frac'] >= STABLE_MIN].copy()
pool = stable if len(stable) else df.copy()
tied = pool[pool['band_ppl'] <= pool['band_ppl'].min() + TIE_BAND].copy()
tied = tied.sort_values(['final_entropy', 'gamma'], ascending=[False, True])
best = tied.iloc[0]

print('=== Sweep table (sorted by band PPL) ===')
print(df.sort_values('band_ppl').to_string(
    index=False,
    columns=['gamma', 'T', 'c1', 'band_ppl', 'best_ppl', 'final_entropy',
             'grad_finite_frac', 'minutes']))

print('\n=== SELECTED (gamma*, T*) ===')
print(f'  gamma* = {best["gamma"]:g}   T* = {best["T"]:g}   '
      f'(band_ppl={best["band_ppl"]:.2f}, entropy={best["final_entropy"]:.3f})')
print('\nPaste these into colab_fock_ostep_langevin_openwebtext.ipynb (Cell 0):')
print(f'  LANGEVIN_GAMMA = {best["gamma"]:g}')
print(f'  LANGEVIN_T     = {best["T"]:g}')
print('  LANGEVIN_TIE_T_TO_BETA = False   # T is now a swept value, not tied')

# Heatmaps: band PPL and final entropy over the (gamma, T) grid.
gammas = sorted(df['gamma'].unique())
Ts     = sorted(df['T'].unique())
def _grid(col):
    M = np.full((len(Ts), len(gammas)), np.nan)
    for _, r in df.iterrows():
        M[Ts.index(r['T']), gammas.index(r['gamma'])] = r[col]
    return M

fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
for ax, col, title, cmap in [
    (axes[0], 'band_ppl', 'val PPL band (lower is better)', 'viridis_r'),
    (axes[1], 'final_entropy', 'predictive entropy (nats)', 'magma'),
]:
    M = _grid(col)
    im = ax.imshow(M, origin='lower', aspect='auto', cmap=cmap)
    ax.set_xticks(range(len(gammas))); ax.set_xticklabels([f'{g:g}' for g in gammas])
    ax.set_yticks(range(len(Ts)));     ax.set_yticklabels([f'{t:g}' for t in Ts])
    ax.set_xlabel('gamma'); ax.set_ylabel('T'); ax.set_title(title)
    for i in range(len(Ts)):
        for j in range(len(gammas)):
            if not np.isnan(M[i, j]):
                ax.text(j, i, f'{M[i, j]:.2f}', ha='center', va='center',
                        color='w', fontsize=8)
    ax.scatter([gammas.index(best['gamma'])], [Ts.index(best['T'])],
               marker='*', s=260, edgecolor='k', facecolor='gold', zorder=5)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.suptitle('O-step Langevin (gamma, T) sweep  (star = selected)')
fig.tight_layout()
_png = RESULTS_DIR / 'gammaT_sweep_heatmaps.png'
fig.savefig(_png, dpi=130, bbox_inches='tight')
print(f'\nSaved heatmaps -> {_png}')
plt.show()
